## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- See https://github.com/PathologyDataScience/simple_triton for details on launching the triton server container and mounting the model repository notebook
- This notebook requires installation of `mil` and `histomics_stream`
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount your model repository directory to the triton container
- Load the model (below)

In [1]:
# install large_image with tile sources
!pip install ../../histomics_stream 'large_image[tiff]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton

# install mil
!pip install ray
!pip install pyarrow
!pip install ../../mil

Looking in links: https://girder.github.io/large_image_wheels
Processing /tf/notebooks/histomics_stream
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for histomics-stream: filename=histomics_stream-2.3.0-py3-none-any.whl size=25526 sha256=7bad69a00fd09653e5318125350945b41e43cfca3254d8eac5d9b34f560457e8
  Stored in directory: /tmp/pip-ephem-wheel-cache-orj1fdcb/wheels/4d/92/92/a22ebaf41cdb2b39fb056793fde0f5347a037c4cd09293a37a
Successfully built histomics-stream
  Attempting uninstall: histomics-stream
    Found existing installation: histomics-stream 2.3.0
    Uninstalling histomics-stream-2.3.0:
      Successfully uninstalled histomics-stream-2.3.0
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
Processing /tf/notebooks/simple_triton
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dep

## Run client with no GPUs

If running Triton and the client on the same machine, we want to stop the client tensorflow from consuming GPU resources. By default, TensorFlow maps nearly all available GPU memory.

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

assert(len(tf.config.list_physical_devices('GPU')) == 0)

## Create a histomics stream study

Parameters in this cell are for reading from the whole-slide image (magnification, tile size, tile overlap, mask file).

In [3]:
from mil.io.utils import study

# slide parameters
batch = 64
magnification = 20
tile = 224
overlap = 0
chunk = 224
mask_threshold=0.5
wsi_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.svs"
mask_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.mask.png"

# create a histomics-stream study from a wsi/mask pair
hs_study = study((wsi_path, mask_path),
                 t=(tile, tile),
                 chunk=(tile, tile),
                 target=20,
                 source="exact")

## Create and load model

The function `feature_extractor` can be used to create feature extraction models in the model repository. Note - this cell will take time as the model is downloaded, saved, and loaded into triton. 

Here we generate a model, load the model into triton, and verify that the model state is "READY".

Parameters in this stage include the inference server (address), the model (model name, maximum batch size).

In [4]:
import json
from google.protobuf.json_format import MessageToDict
import numpy as np
from simple_triton.feature_extraction import feature_extractor
from simple_triton.model import model_config
import tritonclient.grpc as grpcclient

# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server
keras_name = "ConvNeXtXLarge"
model_name = f"{keras_name}.tensorflow"  # set model name

# create the model and capture output dimensionality
if not os.path.exists(os.path.join("/tf/notebooks/models", model_name)):
    dimension_output = feature_extractor("/tf/notebooks/models",
                                         keras_name,
                                         model_name,
                                         t=(tile, tile),
                                         pool="avg")

# create triton client
client = grpcclient.InferenceServerClient(url=url, verbose=True)

# load tensorflow model with larger batch size
config = {'maxBatchSize': 256}
client.load_model(model_name, config=json.dumps(config))

# check readiness
client.get_model_repository_index()

# deleting the client in main prevents conflicts with child process clients
del client

Imported version of grpc is 1.50.0. There is a memory leak in certain Python GRPC versions (1.43.0 to be specific). Please use versions <1.43.0 or >=1.51.1 to avoid leaks (see https://github.com/grpc/grpc/issues/28513).


load_model, metadata ()
override files omitted:
model_name: "ConvNeXtXLarge.tensorflow"
parameters {
  key: "config"
  value {
    string_param: "{\"maxBatchSize\": 256}"
  }
}

Loaded model 'ConvNeXtXLarge.tensorflow'
get_model_repository_index, metadata ()

models {
  name: ".ipynb_checkpoints"
}
models {
  name: "ConvNeXtXLarge.tensorflow"
  version: "1"
  state: "READY"
}
models {
  name: "EfficientNetV2L.tensorflow"
  version: "1"
  state: "READY"
}
models {
  name: "EfficientNetV2S.onnx"
}
models {
  name: "EfficientNetV2S.tensorflow"
  version: "1"
  state: "READY"
}
models {
  name: "EfficientNetV2S_TRT_FP16.tensorflow"
}
models {
  name: "EfficientNetV2S_TRT_FP32.tensorflow"
}



## Run the inference

Parameters here include the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [5]:
from simple_triton.feature_extraction import histomics_stream_inference
from simple_triton.submitter import analyze
import time

# inference parameters
batch = 64
limit = 10  # limit on number of pending requests per worker
workers = 32  # total number of Submitter workers
verbose = True  # set verbose as False

# start timer
start = time.time()

# inference
features, tile_info, times = histomics_stream_inference(hs_study,
                                                        model_name,
                                                        url="localhost:8001",
                                                        batch=batch,
                                                        workers=workers,
                                                        limit=limit)

# display elapsed time
print(f"Total elapsed time: {time.time()-start}")

# analyze performance
analyze(times)

Total elapsed time: 32.09034061431885
                             median    min    max
-------------------------  --------  -----  -----
total (sec)                   17.66   6.48  31.37
qin (% total)                 11.23   1.85  33.21
qout (% total)                 0.02   0.01   0.09
in-process (% total)          88.75  66.75  98.14
completion (% in-process)     99.59  87.59  99.92
retrieval (% in-process)       0.04   0.00   0.17
other (% in-process)           0.35   0.08  12.35


## Write features to .tfr

In [6]:
from mil.io.reader import read_record, peek
from mil.io.writer import write_record
import tensorflow as tf

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record("./triton.tfr", features, tile_info, labels, structured=False, precision=tf.float16)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)

(<tf.Tensor: shape=(4855, 2048), dtype=float16, numpy=
 array([[-0.7363 ,  0.06683,  0.0635 , ...,  1.357  , -1.75   , -0.834  ],
        [ 1.208  , -0.061  , -0.09955, ...,  1.31   , -1.022  , -0.8955 ],
        [ 1.232  , -0.1411 ,  0.3655 , ...,  3.213  , -1.374  , -0.8115 ],
        ...,
        [ 0.5293 , -0.1107 ,  0.6147 , ...,  5.555  , -2.523  , -0.1669 ],
        [ 0.755  , -0.08887,  0.5156 , ...,  6.05   , -2.428  , -0.05554],
        [ 0.4563 , -0.00912,  0.6597 , ...,  6.113  , -1.8545 , -0.4365 ]],
       dtype=float16)>,
 {'labels': <tf.Tensor: shape=(10,), dtype=float32, numpy=
  array([0.32636747, 0.4093263 , 0.9996222 , 0.98195666, 0.5823284 ,
         0.7684934 , 0.71833754, 0.3656233 , 0.59421176, 0.902352  ],
        dtype=float32)>},
 {'filename': <tf.Tensor: shape=(1,), dtype=string, numpy=
  array([b'/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.svs'],
        dtype=object)>,
  'level': <tf.Tensor: shape=(1,), dtype=int64, numpy=arr